# Plugin registry — quick demo

This compact notebook demonstrates `biolm_utils.plugin_registry` basics: register, apply, restore, and unregister plugins in a small, testable example.

The registry stores factory callables that return a `Config`-like mapping; `apply_plugin` sets the current configuration via `set_config` and pushes the previous state onto an internal stack so it can be restored later.

In [ ]:
# Basic imports and helpers
from biolm_utils.plugin_registry import (
    register_plugin, list_plugins, apply_plugin,
    get_current_plugin, get_applied_stack, unregister_plugin,
    restore_previous_plugin
)
from biolm_utils.config import get_config

def print_state(label=''):
    print('---', label)
    print('registered:', list_plugins())
    print('active:', get_current_plugin())
    print('stack:', [n for n,_ in get_applied_stack()])
    try:
        cfg = get_config()
        print('learning_rate:', getattr(cfg, 'learning_rate', None))
    except Exception:
        print('config: (not set)')
    print()

In [ ]:
# Register a simple demo plugin 'demo_a' that returns a minimal config mapping
def demo_factory_a():
    return {
        'model_cls_for_pretraining': 'preA',
        'model_cls_for_finetuning': 'finA',
        'tokenizer_cls': 'tokA',
        'learning_rate': 1e-3,
        'max_grad_norm': 1.0,
        'weight_decay': 0.0,
        'special_tokenizer_for_trainer_cls': None,
        'datacollator_cls_for_pretraining': None,
        'datacollator_cls_for_finetuning': None,
        'add_special_tokens': False,
        'config_cls': None,
        'pretraining_required': False,
        'dataset_cls': None,
    }

register_plugin('demo_a', demo_factory_a)
print_state('after register demo_a')

In [ ]:
# Apply demo_a and inspect the active config
apply_plugin('demo_a')
print_state('after apply demo_a')

# Read value from the active config via get_config()
print('active learning_rate =', get_config().learning_rate)

In [ ]:
# Register and apply a second plugin 'demo_b'
def demo_factory_b():
    return {**demo_factory_a(), 'learning_rate': 1e-2}

register_plugin('demo_b', demo_factory_b)
apply_plugin('demo_b')
print_state('after register+apply demo_b')
print('stack (names):', [n for n,_ in get_applied_stack()])

In [ ]:
# Restore to the previous plugin (should go back to demo_a)
restore_previous_plugin()
print_state('after restore_previous_plugin')
print('current plugin:', get_current_plugin())

In [ ]:
# Unregister demo_b and show that applying it raises an error
unregister_plugin('demo_b')
print_state('after unregister demo_b')
try:
    apply_plugin('demo_b')
except Exception as e:
    print('applying demo_b failed ->', type(e).__name__, e)

In [ ]:
# Final state: list plugins and applied stack (names and non-empty configs)
print_state('final')
print('applied stack details:')
for name, cfg in get_applied_stack():
    print('-', name, 'config_present=', cfg is not None)

In [ ]:
# Quick assertions (lightweight smoke checks)
assert 'demo_a' in list_plugins()
assert get_current_plugin() == 'demo_a'
print('Smoke checks passed')

In [ ]:
# End of demo — unregister demo_a and reset (optional cleanup)
try:
    unregister_plugin('demo_a')
except Exception:
    pass
restore_previous_plugin()  # ensure stack is clean
print('demo completed')